### Chicago 2025 World Championships Race Analysis

This notebook contains an analysis of the 2025 World Championships in Chicago. In this first section, we load the data and get our first look at its characteristics.

In [ ]:
import pandas as pd
from pathlib import Path

data_path = Path.cwd() / ".." / ".." / "data" / "results.csv"
if not data_path.is_file():
    raise ValueError(f"results not found at path {data_path}")

df = pd.read_csv(data_path)
df.head()

In [ ]:
# exclude values where we couldn't scrape splits
df = df[df["has_splits"]]
print(f"{len(df)} results after filtering those without splits")

In [ ]:
import pyrox.models as models

# number of elite men
n_elite = (df["division_name"] == str(models.DivisionName.ELITE_MEN)).sum()
print(f"{n_elite} elite men")

# number of solo pro men
n_non = (df["division_name"] == str(models.DivisionName.PRO_MEN)).sum()
print(f"{n_non} non-elite pro solo men")

**Pace Analysis**

In this analysis, we look at the mean and variance of run and station duration for elites and non-elites, and compare the results between the two groups.

In [ ]:
# a dataframe of just the elites
elites = df[df["division_name"] == str(models.DivisionName.ELITE_MEN)]
# a dataframe of the just the non-elites, limited to top 100 finishers
non_elites = df[df["division_name"] == str(models.DivisionName.PRO_MEN)].head(256)

In [ ]:
import numpy as np

# the column names for the runs
run_cols = [f"run_{i+1}" for i in range(8)]
# the column names for the stations
station_cols = [str(station) for station in models.Station]

# the elite run splits as numpy array
elite_runs = elites[run_cols].to_numpy()
# the elite stations as numpy array
elite_stations = elites[station_cols].to_numpy()

# the non-elite run splits as numpy array
non_runs = non_elites[run_cols].to_numpy()
# the non-elite stations as numpy array
non_stations = non_elites[station_cols].to_numpy()

assert all(
    a.shape[1] == 8 for a in [elite_runs, elite_stations, non_runs, non_stations]
)


# finally, we concatenate to combine all events for both groups;
# e.g. run 1, run 2, ... run 8, ski, sled push, etc...
elites_np = np.concatenate((elite_runs, elite_stations), axis=1)
non_np = np.concatenate((non_runs, non_stations), axis=1)

assert all(a.shape[1] == 16 for a in [elites_np, non_np])

In [ ]:
import numpy as np
import numpy.typing as npt


def mean_and_std(a: npt.NDArray) -> tuple[npt.NDArray, npt.NDArray]:
    """Compute column-wise mean and variance."""
    return np.mean(a, axis=0), np.std(a, axis=0)


# compute mean and standard deviation for elite and non-elite races
elite_means, elite_stds = mean_and_std(elites_np)
non_means, non_stds = mean_and_std(non_np)

In [ ]:
# now, interleave to get events for groups in race order;
# e.g. run 1, ski, run 2, sled push, etc.
from typing import Any, Iterable

def interleave(a: Iterable[Any], b: Iterable[Any]) -> list[Any]:
    return [item for pair in zip(a, b) for item in pair]

# means race order, elite and stds race order, elite
mro_e, sro_e = np.array(interleave(elite_means[:8], elite_means[8:])), np.array(interleave(elite_stds[:8], elite_stds[8:]))
# means race order, non and stds race order, non
mro_n, sro_n = np.array(interleave(non_means[:8], non_means[8:])), np.array(interleave(non_stds[:8], non_stds[8:]))

In [ ]:
from plotnine import (
    ggplot,
    aes,
    element_text,
    labs,
    theme,
    position_dodge,
    geom_segment,
    geom_point,
    scale_y_continuous,
)
from typing import Any


def interleave(a: list[Any], b: list[Any]) -> list[Any]:
    assert len(a) == len(b), "broken precondition"
    return [item for pair in zip(a, b) for item in pair]

def format_seconds_to_mmss(breaks):
    return [f"{int(s//60)}:{int(s%60):02d}" for s in breaks]


def make_plot(
    elite_means: npt.NDArray,
    elite_stds: npt.NDArray,
    non_means: npt.NDArray,
    non_stds: npt.NDArray,
) -> ggplot:
    """Make a plot."""
    # generate the final order of labels
    labels = interleave(
        [f"run_{i+1}" for i in range(8)], [str(station) for station in models.Station]
    )
    # make the labels pretty
    labels = [
        " ".join(map(str.capitalize, e.split("_"))) for e in labels
    ]

    # compute lower and upper for elites and non-elites
    elite_lower = [int(mean - std) for mean, std in zip(elite_means, elite_stds)]
    elite_upper = [int(mean + std) for mean, std in zip(elite_means, elite_stds)]
    non_lower = [int(mean - std) for mean, std in zip(non_means, non_stds)]
    non_upper = [int(mean + std) for mean, std in zip(non_means, non_stds)]

    # interleave to get final data for df
    mean = interleave([int(v) for v in elite_means.astype(int)], [int(v) for v in non_means.astype(int)])
    lower = interleave([v for v in elite_lower], [v for v in non_lower])
    upper = interleave([v for v in elite_upper], [v for v in non_upper])

    df = pd.DataFrame(
        {
            # interleave with self to make 2x copies of each event
            "event": interleave(labels, labels),
            "division": ["elite", "non-elite"] * len(elite_means),
            "mean": mean,
            "lower": lower,
            "upper": upper,
        }
    )
    df["event"] = pd.Categorical(df["event"], categories=labels, ordered=True)

    # dodge so classes are side-by-side instead of overlapping
    d = position_dodge(width=0.4)

    p = (
        ggplot(df, aes('event', 'mean', color='division'))
        # vertical line: lower -> upper
        + geom_segment(
            aes(x='event', xend='event', y='lower', yend='upper'),
            position=d
        )
        # horizontal whisker (lower)
        + geom_segment(
            aes(x='event', xend='event', y='lower', yend='lower'),
            position=d,
            size=2
        )
        # horizontal whisker (upper)
        + geom_segment(
            aes(x='event', xend='event', y='upper', yend='upper'),
            position=d,
            size=2
        )
        # mean point
        + geom_point(position=d, size=3)
        + theme(figure_size=(14, 6), axis_text_x=element_text(rotation=45, ha="right"))
        + labs(x="Event", y="Duration (mm:ss)", title="Elites versus Top-K Non-Elite Pro, Chicago 2025")
        + scale_y_continuous(labels=format_seconds_to_mmss)
    )
    return p

make_plot(mro_e, sro_e, mro_n, sro_n)